In [ ]:
from pathlib import Path
import copy
import json
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import DataLoader

SEED = 29
NODE_DATA_LEN = 33

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path(".").resolve()
DATA_DIR = ROOT / "model_data"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"ROOT={ROOT}")
print(f"DATA_DIR={DATA_DIR}")
print(f"device={DEVICE}")


In [ ]:
train_20 = np.load(DATA_DIR / "train.npy", allow_pickle=True)
test_20 = np.load(DATA_DIR / "test.npy", allow_pickle=True)
val_20 = np.load(DATA_DIR / "val.npy", allow_pickle=True)

with open(DATA_DIR / "scalers.pkl", "rb") as f:
    SCALERS = pickle.load(f)
TARGET_SCALER = SCALERS["target_scaler"]

manifest_path = DATA_DIR / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        MANIFEST = json.load(f)
else:
    MANIFEST = {}


def unpack_predictor_unit(unit, device):
    node = torch.as_tensor(unit["nodes"], dtype=torch.float32).reshape(
        -1, NODE_DATA_LEN
    )
    if node.shape[0] <= 1:
        return None
    y = torch.as_tensor(unit["y"], dtype=torch.float32).reshape(-1)
    return node.to(device), y.to(device)


def split_graph_counts(split):
    sizes = np.asarray([
        np.asarray(split[i]["nodes"]).reshape(-1, NODE_DATA_LEN).shape[0]
        for i in range(len(split))
    ])
    return {
        "all_graphs": int(sizes.size),
        "multi_zone_graphs": int((sizes > 1).sum()),
        "one_zone_graphs": int((sizes <= 1).sum()),
        "retained_nodes": int(sizes[sizes > 1].sum()),
    }


In [ ]:
class RMSELoss(nn.Module):
    def forward(self, pred, target):
        return torch.sqrt(nn.functional.mse_loss(pred, target) + 1e-6)


class Predictor(nn.Module):

    def __init__(self, in_size=NODE_DATA_LEN, layer_size=64, layer_num=4):
        super().__init__()
        self.input_layer = nn.Linear(in_size, layer_size)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(layer_size, layer_size) for _ in range(layer_num)]
        )
        self.output_layer = nn.Linear(layer_size, 1)
        self.activation = nn.ReLU()
        self.norms = nn.ModuleList(
            [nn.LayerNorm(layer_size) for _ in range(layer_num + 1)]
        )

    def forward(self, x):
        x = self.norms[0](self.activation(self.input_layer(x)))
        for i, layer in enumerate(self.hidden_layers):
            x = self.norms[i + 1](self.activation(layer(x)))
        return self.output_layer(x).squeeze(-1)


def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


def _physical_rmse(scaler, preds, targets):
    p = scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    t = scaler.inverse_transform(targets.reshape(-1, 1)).flatten()
    return float(np.sqrt(np.mean((p - t) ** 2)))


def inverse_target(values):
    arr = np.asarray(values, dtype=np.float64).reshape(-1, 1)
    return TARGET_SCALER.inverse_transform(arr).ravel()


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    preds, targets = [], []

    for unit in loader:
        parsed = unpack_predictor_unit(unit, DEVICE)
        if parsed is None:
            continue
        node, y = parsed
        preds.append(model(node).flatten().cpu())
        targets.append(y.cpu())

    if not preds:
        raise RuntimeError("No multi-zone graphs were available for evaluation.")

    preds = torch.cat(preds)
    targets = torch.cat(targets)
    preds_np = preds.numpy()
    targets_np = targets.numpy()
    rmse = loss_fn(preds, targets).item()
    r2 = r2_score(targets_np, preds_np)
    return {
        "rmse": rmse,
        "r2": r2,
        "preds": preds_np,
        "targets": targets_np,
    }


def plot_predictions(preds, targets, title):
    preds_phys = inverse_target(preds)
    targets_phys = inverse_target(targets)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(preds_phys, targets_phys, s=12, alpha=0.5, color="darkorange")
    lo = min(preds_phys.min(), targets_phys.min())
    hi = max(preds_phys.max(), targets_phys.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="green")
    ax.set_xlabel("Predictions")
    ax.set_ylabel("Ground Truth")
    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    plt.show()
    return preds_phys, targets_phys


In [ ]:
def flatten_split(split):
    xs, ys = [], []
    for i in range(len(split)):
        node = np.asarray(split[i]["nodes"]).reshape(-1, NODE_DATA_LEN)
        if node.shape[0] <= 1:
            continue
        ys.append(np.asarray(split[i]["y"]).reshape(-1))
        xs.append(node)
    X = torch.as_tensor(np.concatenate(xs, axis=0), dtype=torch.float32)
    Y = torch.as_tensor(np.concatenate(ys, axis=0), dtype=torch.float32)
    return X, Y


TRAIN_X, TRAIN_Y = flatten_split(train_20)
VAL_X, VAL_Y = flatten_split(val_20)
TEST_X, TEST_Y = flatten_split(test_20)
print(f"flattened nodes -> train={TRAIN_X.shape[0]} "
      f"val={VAL_X.shape[0]} test={TEST_X.shape[0]}")


@torch.no_grad()
def evaluate_flat(model, X, Y, loss_fn, eval_batch=8192):
    model.eval()
    device = next(model.parameters()).device
    preds = []
    for i in range(0, X.shape[0], eval_batch):
        preds.append(model(X[i:i + eval_batch].to(device)).flatten().cpu())
    preds = torch.cat(preds)
    preds_np = preds.numpy()
    targets_np = Y.numpy()
    rmse = loss_fn(preds, Y).item()
    r2 = r2_score(targets_np, preds_np)
    return {
        "rmse": rmse,
        "r2": r2,
        "preds": preds_np,
        "targets": targets_np,
    }


BASE_CONFIG = {
    "seed": SEED,
    "node_data_len": NODE_DATA_LEN,
    "predictor_layer_num": 4, 
    "weight_decay": 1e-5,
    "epochs": 200,
    "lr_step_size": 10,
    "lr_gamma": 0.99,
    "min_delta": 1e-5,
    "learning_rate": 0.013819399545084802,
    "node_batch_size": 128,
    "predictor_layer_size": 64,
    "use_wandb": False,
}

LAST_MODEL = None
LAST_RESULTS = None
LAST_HISTORY = None


NODE_SWEEP_CONFIG = {
    "method": "random",
    "metric": {"name": "val_rmse", "goal": "minimize"},
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-7,
            "max": 1e-1,
        },
        "node_batch_size": {"values": [32, 64, 128, 256, 512, 1024, 2048]},
        "predictor_layer_size": {"values": [64, 96, 128]},
    },
}


def _wandb_log(cfg, data):
    if cfg.get("use_wandb"):
        import wandb
        wandb.log(data)


def config_from_wandb(sampled, base):
    cfg = {**base, "use_wandb": True}
    cfg["learning_rate"] = float(sampled.learning_rate)
    cfg["node_batch_size"] = int(sampled.node_batch_size)
    cfg["predictor_layer_size"] = int(sampled.predictor_layer_size)
    return cfg


def train_mlp_nodes(cfg=None):
    global LAST_MODEL, LAST_RESULTS, LAST_HISTORY

    cfg = {**BASE_CONFIG, **(cfg or {})}

    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg["seed"])

    train_ds = torch.utils.data.TensorDataset(TRAIN_X, TRAIN_Y)
    loader_gen = torch.Generator().manual_seed(cfg["seed"])
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["node_batch_size"],
        shuffle=True,
        generator=loader_gen,
    )

    model = Predictor(
        in_size=cfg["node_data_len"],
        layer_size=cfg["predictor_layer_size"],
        layer_num=cfg["predictor_layer_num"],
    ).to(DEVICE)
    model.apply(init_weights)

    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[setup] trainable parameters = {parameter_count}")
    _wandb_log(cfg, {"parameter_count": parameter_count})

    loss_fn = RMSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=cfg["lr_step_size"],
        gamma=cfg["lr_gamma"],
    )

    best_val = float("inf")
    best_state = None
    history = {"train_rmse": [], "val_rmse": [], "val_r2": []}

    for epoch in range(cfg["epochs"]):
        model.train()
        batch_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb).flatten()
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        scheduler.step()
        train_rmse = float(np.mean(batch_losses))
        val_metrics = evaluate_flat(model, VAL_X, VAL_Y, loss_fn)

        history["train_rmse"].append(train_rmse)
        history["val_rmse"].append(val_metrics["rmse"])
        history["val_r2"].append(val_metrics["r2"])

        print(
            f"[epoch {epoch + 1:02d}/{cfg['epochs']}] "
            f"lr={scheduler.get_last_lr()[0]:.2e} "
            f"train_rmse={train_rmse:.4f} "
            f"val_rmse={val_metrics['rmse']:.4f} "
            f"val_r2={val_metrics['r2']:.4f}"
        )
        _wandb_log(cfg, {
            "epoch": epoch,
            "lr": scheduler.get_last_lr()[0],
            "train_rmse": train_rmse,
            "val_rmse": val_metrics["rmse"],
            "val_r2": val_metrics["r2"],
        })

        if val_metrics["rmse"] < best_val - cfg["min_delta"]:
            best_val = val_metrics["rmse"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    print("\n[post] final evaluation on best checkpoint")
    results = {}
    for name, (X, Y) in (
        ("train", (TRAIN_X, TRAIN_Y)),
        ("val", (VAL_X, VAL_Y)),
        ("test", (TEST_X, TEST_Y)),
    ):
        metrics = evaluate_flat(model, X, Y, loss_fn)
        phys = _physical_rmse(TARGET_SCALER, metrics["preds"], metrics["targets"])
        metrics["rmse_phys"] = phys
        results[name] = metrics
        print(
            f"  {name:5s}: R2={metrics['r2']:.4f} RMSE={metrics['rmse']:.4f} "
            f"rmse(phys)={phys:.3f}"
        )
        _wandb_log(cfg, {
            f"{name}_r2_final": metrics["r2"],
            f"{name}_rmse_final": metrics["rmse"],
            f"{name}_rmse_phys": phys,
        })

    test_r2 = results["test"]["r2"]
    test_rmse = results["test"]["rmse"]
    ckpt = MODEL_DIR / (
        f"mlp_R2_{test_r2:.4f}"
        f"_RMSE_{test_rmse:.4f}.pt"
    )
    torch.save(
        {
            "model_state": model.state_dict(),
            "config": cfg,
            "metrics": {
                name: {
                    "r2": metrics["r2"],
                    "rmse": metrics["rmse"],
                    "rmse_phys": metrics["rmse_phys"],
                }
                for name, metrics in results.items()
            },
            "manifest_seed": MANIFEST.get("seed"),
            "batching": "shuffled_nodes",
        },
        ckpt,
    )
    print("saved:", ckpt)

    LAST_MODEL = model
    LAST_RESULTS = results
    LAST_HISTORY = history
    return model, results, history


def train_sweep():
    import wandb
    wandb.init()
    try:
        cfg = config_from_wandb(wandb.config, BASE_CONFIG)
        wandb.config.update(
            {f"baseline_cfg/{key}": value for key, value in cfg.items()},
            allow_val_change=True,
        )
        train_mlp_nodes(cfg)
    finally:
        wandb.finish()


In [ ]:
cfg = {**BASE_CONFIG, "use_wandb": False}
model, results, history = train_mlp_nodes(cfg)

In [ ]:
import wandb
wandb.login()

node_sweep_id = wandb.sweep(
    NODE_SWEEP_CONFIG,
    project="",
    entity="",
)
print("node_sweep_id:", node_sweep_id)
wandb.agent(node_sweep_id, function=train_sweep, count=20)


In [ ]:
def load_checkpoint(path, device=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if "config" not in ckpt or "model_state" not in ckpt:
        raise ValueError(
            "Expected a bundled checkpoint containing config and model_state"
        )

    cfg = dict(ckpt["config"])
    dev = torch.device(device or DEVICE)
    model = Predictor(
        in_size=cfg["node_data_len"],
        layer_size=cfg["predictor_layer_size"],
        layer_num=cfg["predictor_layer_num"],
    ).to(dev)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, cfg


def plot_regression(preds, targets, title, scaler=None, save_path=None):
    preds = np.asarray(preds).reshape(-1)
    targets = np.asarray(targets).reshape(-1)
    if scaler is not None:
        preds = scaler.inverse_transform(preds.reshape(-1, 1)).ravel()
        targets = scaler.inverse_transform(targets.reshape(-1, 1)).ravel()
        xlabel, ylabel = "Predicted (phys)", "Ground truth (phys)"
    else:
        xlabel, ylabel = "Predicted (scaled)", "Ground truth (scaled)"

    lo = float(min(preds.min(), targets.min()))
    hi = float(max(preds.max(), targets.max()))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(preds, targets, s=12, alpha=0.45, color="darkorange")
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="green", label="x = y")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.25)
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=120)
        print(f"[reproduce] saved plot -> {save_path}")
    plt.show()
    return fig


def reproduce_from_checkpoint(path, device=None, plot=True, physical=True,
                              save_plots=True):
    model, cfg = load_checkpoint(path, device)
    loss_fn = RMSELoss()
    splits = {
        "train": (TRAIN_X, TRAIN_Y),
        "val": (VAL_X, VAL_Y),
        "test": (TEST_X, TEST_Y),
    }

    print(f"[reproduce] loaded {path}")
    print(f"[reproduce] evaluating on device={next(model.parameters()).device}")
    results = {}
    plot_dir = None
    if plot and save_plots:
        plot_dir = MODEL_DIR / "reproduce_plots"
        plot_dir.mkdir(parents=True, exist_ok=True)

    for name, (X, Y) in splits.items():
        metrics = evaluate_flat(model, X, Y, loss_fn)
        phys = _physical_rmse(TARGET_SCALER, metrics["preds"], metrics["targets"])
        print(
            f"  {name:5s}: R2={metrics['r2']:.4f} RMSE={metrics['rmse']:.4f} "
            f"rmse(phys)={phys:.3f}"
        )
        results[name] = {
            "r2": metrics["r2"],
            "rmse": metrics["rmse"],
            "rmse_phys": phys,
            "preds": metrics["preds"],
            "targets": metrics["targets"],
        }

        if plot:
            unit = "phys" if physical else "scaled"
            title = f"{name}  R2={metrics['r2']:.4f}  RMSE={metrics['rmse']:.4f}"
            if physical:
                title += f"  RMSE(phys)={phys:.3f}"
            save_path = None
            if plot_dir is not None:
                save_path = str(plot_dir / f"{name}_regression_{unit}.png")
            plot_regression(
                metrics["preds"],
                metrics["targets"],
                title=title,
                scaler=TARGET_SCALER if physical else None,
                save_path=save_path,
            )

    return results, model, cfg


In [ ]:
reproduce_from_checkpoint(
    "./models/mlp_R2_0.9272_RMSE_0.2773.pt",
    save_plots=False,
    physical=False
)